# Module 1 — Exercise Solutions

---

## Exercise 1: Evaluate a New Dataset — Solution

In [ ]:
!pip install -q datasets pandas

In [ ]:
from datasets import load_dataset
import re

ds = load_dataset("Amod/mental_health_counseling_conversations", split="train")
print(f"Dataset size: {len(ds):,}")
print(f"Columns: {ds.column_names}")

# Display 10 random examples
import random
random.seed(42)
indices = random.sample(range(len(ds)), 10)
for i in indices[:3]:  # Show 3 for readability
    print(f"\n{'='*60}")
    print(f"Context: {ds[i]['Context'][:200]}...")
    print(f"Response: {ds[i]['Response'][:200]}...")

In [ ]:
# Audit for quality issues
total = len(ds)

# 1. Persona contamination
persona_patterns = [r"as a therapist", r"as a counselor", r"in my practice",
                    r"my client", r"my patient"]
persona_count = sum(1 for ex in ds if any(re.search(p, ex["Response"].lower()) for p in persona_patterns))

# 2. Boilerplate
boilerplate_patterns = [r"thank you for sharing", r"i appreciate you",
                        r"it takes courage", r"i hear you"]
boilerplate_count = sum(1 for ex in ds if any(re.search(p, ex["Response"].lower()) for p in boilerplate_patterns))

# 3. Safety disclaimers
safety_patterns = [r"professional help", r"therapist", r"counselor",
                   r"crisis", r"hotline", r"988", r"emergency"]
safety_count = sum(1 for ex in ds if any(re.search(p, ex["Response"].lower()) for p in safety_patterns))

# 4. Very short
short_count = sum(1 for ex in ds if len(ex["Response"]) < 50)

print(f"Persona contamination: {persona_count:,} ({persona_count/total*100:.1f}%)")
print(f"Boilerplate phrases:   {boilerplate_count:,} ({boilerplate_count/total*100:.1f}%)")
print(f"Safety disclaimers:    {safety_count:,} ({safety_count/total*100:.1f}%)")
print(f"Very short (<50ch):    {short_count:,} ({short_count/total*100:.1f}%)")

**Assessment:**

The dataset quality depends on what patterns you find. Key things to check:
- If persona contamination is high, the model will learn to roleplay as a specific therapist
- If safety disclaimers are low, the model won't learn to recommend professional help for serious issues
- Mental health has HIGHER safety requirements than general medical Q&A — missing crisis resources is dangerous

---

## Exercise 2: Dataset Size vs Quality — Solution

In [ ]:
from datasets import load_dataset
import re

ds = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")
print(f"Original size: {len(ds):,}")

PERSONA_PATTERNS = [
    r"chat\s*doctor", r"welcome to", r"thanks for",
    r"posting (?:your|the) query", r"i understand your concern"
]

BOILERPLATE_PATTERNS = [
    r"hope this helps", r"wishing you", r"good luck",
    r"wish you good health", r"take care"
]

def is_clean(example):
    output = example["output"].lower()
    has_persona = any(re.search(p, output) for p in PERSONA_PATTERNS)
    has_boilerplate = any(re.search(p, output) for p in BOILERPLATE_PATTERNS)
    return not has_persona and not has_boilerplate

clean_ds = ds.filter(is_clean)
print(f"Clean examples: {len(clean_ds):,} ({len(clean_ds)/len(ds)*100:.1f}%)")
print(f"Removed: {len(ds) - len(clean_ds):,} ({(len(ds) - len(clean_ds))/len(ds)*100:.1f}%)")

**Answers:**

1. After removing persona contamination (63.1%): ~41,000 remain
2. After also removing boilerplate (with overlap): actual count from filter above
3. Yes — a filtered 30-40K dataset with only clean examples would train a better model
   than the full 112K with 63% contamination. The model would learn medical content
   without also learning "Hi welcome to Chat Doctor" greetings.
4. Actual count: see filter output above

---

## Exercise 3: Reformatting Prompt — Solution

In [ ]:
SAMPLE_INPUT = """
Question: What causes croup in children?
Answer: Croup is most commonly caused by parainfluenza viruses, particularly 
types 1 and 3. Other causes include influenza A and B, adenovirus, respiratory 
syncytial virus (RSV), and measles. The infection causes swelling of the larynx, 
trachea, and bronchi. Peak incidence is between 6 months and 3 years of age, 
with most cases occurring in autumn.
"""

REFORMATTING_PROMPT = """
You are reformatting medical text into parent-friendly answers for a
pediatrics helpline chatbot.

Rules:
- Use a warm, reassuring tone. Parents calling are worried.
- Write at a 6th-grade reading level. Avoid jargon — if you must use
  a medical term, explain it in parentheses.
- Always include WHEN TO CALL THE DOCTOR and WHEN TO GO TO THE ER
  as separate clearly labeled sections.
- End with: "Remember, you know your child best. If something doesn't
  feel right, call your pediatrician — that's what they're there for."
- Keep the answer concise and focused. Do not pad with filler.
- Do NOT diagnose. Frame as "common causes include..." not "your child has..."
"""

print("Sample prompt designed.")
print("Key design choices:")
print("  - Warm tone (not clinical)")
print("  - 6th-grade reading level")
print("  - Clear urgency tiers (doctor vs ER)")
print("  - Safety disclaimer built into format")
print("  - No diagnosis language")

**Expected output for croup question:**

```
Croup is a common childhood illness that makes your child's voice sound
hoarse and gives them a barking cough (it can sound scary, but it's
usually manageable at home).

Common causes include:
- Parainfluenza virus (the most common cause)
- Flu viruses
- RSV (respiratory syncytial virus — a common cold virus in babies)

It's most common in children between 6 months and 3 years old, and
tends to show up more in the fall.

WHEN TO CALL YOUR PEDIATRICIAN:
- The barking cough lasts more than 3 days
- Your child develops a fever over 103.5°F
- Your child seems unusually tired or fussy

WHEN TO GO TO THE ER:
- Your child is struggling to breathe or making a high-pitched
  whistling sound when breathing in (called stridor)
- Your child's lips or fingernails look bluish
- Your child is drooling and can't swallow

Remember, you know your child best. If something doesn't feel right,
call your pediatrician — that's what they're there for.
```

---

## Exercise 4: System Prompt Ablation — Solution

In [ ]:
# Solution prompts:

SAFETY_FIRST_PROMPT = """You are a healthcare information assistant. Your TOP PRIORITY is safety.

Rules (in order of importance):
1. ALWAYS start with: "This is general health information, not medical advice."
2. ALWAYS end with: "Please consult a healthcare professional for personalized guidance."
3. For ANY symptom that could be an emergency (chest pain, stroke signs, difficulty breathing),
   say "Call 911 or go to the nearest emergency room immediately" BEFORE any other information.
4. Never say "you should take" or "you need" — instead say "your doctor may recommend"
5. Include general health information, but frame everything as possibilities, not diagnoses.
"""

STRUCTURED_PROMPT = """You are a healthcare information assistant. Format EVERY response with:

## Overview
One-paragraph summary.

## Key Points
- Bullet point 1
- Bullet point 2
- (3-5 bullet points)

## What to Know
Numbered list of important details:
1. First detail
2. Second detail

## When to See a Doctor
Specific warning signs that need medical attention.

Always use markdown headers, bullet points, and numbered lists.
Keep paragraphs to 2-3 sentences maximum.
"""

print("Safety-first prompt: prioritizes disclaimers, frames as possibilities")
print("Structured prompt: enforces consistent formatting with headers/lists")
print()
print("Expected findings:")
print("  - Safety-first produces more disclaimers but can feel overly cautious")
print("  - Structured produces better-organized output but may miss safety cues")
print("  - Neither fully matches fine-tuning because the model doesn't consistently")
print("    follow complex prompts — fine-tuning bakes behavior into weights")

**Key Insight:**

Prompt engineering can get you 70-80% of the way for a capable base model.
Fine-tuning's advantage is CONSISTENCY — the model follows the pattern on
every response, not just when the prompt is perfect. For production healthcare
applications, that consistency matters more than the marginal quality improvement.